# Qwen3-8B ecological-dilemma fine-tuning

This notebook can fine-tune `Qwen/Qwen3-8B` on one of three 98-case ecological-versus-human dilemma arms: `prompt_only`, `ecological_option`, or `human_option`. Its default path now discovers and hash-verifies the already completed checkpoint for every arm before evaluation. The prompt-only arm retains the original user-only causal objective. The two answer arms pair each dilemma with the exact corresponding option field as a one-turn assistant response, mask the user turn, and apply loss only to that option text and its terminating token. No arm contains a rationale.

The primary follow-up evaluates all three saved adapters on a supervision-matched battery: reversed-polarity `Yes`/`No`, counterbalanced `A`/`B`, and complete option-text scoring in both display orders. Each verified bundle is saved beneath its source run on Google Drive and published to its arm-specific GitHub result folder. The older eight forward-polarity prompts remain available as a cached reference; the six earlier controls remain in commented cells but are no longer run by default.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
# Colab's optional TorchAO build can conflict with PEFT. This workflow uses
# ordinary BF16 LoRA and does not use TorchAO quantization.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
REPOSITORY_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This workflow requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

## Configuration

The defaults retain the proven H4rmony Qwen LoRA setup where it transfers: BF16, rank 16, alpha 32, dropout 0.05, all linear layers, three epochs, micro-batch size 1, gradient accumulation 16, and seed 42. `TRAINING_ARM` selects the arm only if you deliberately force a retrain. Evaluation requires compatible completed runs for all three arms; dataset paths and isolated local/Drive roots follow automatically.

In [ ]:
from google.colab import userdata
from scripts.ecological_prompt_sft import (
    DEFAULT_DATASET_PATHS,
    TRAINING_ARMS,
    DilemmaSFTConfig,
)

TRAINING_ARM = "ecological_option"  # prompt_only | ecological_option | human_option
assert TRAINING_ARM in TRAINING_ARMS
EVALUATION_ARMS = tuple(TRAINING_ARMS)
OUTPUT_SLUGS = {
    "prompt_only": "ecological_dilemma_prompt_qwen3_8b",
    "ecological_option": "ecological_dilemma_ecological_option_qwen3_8b",
    "human_option": "ecological_dilemma_human_option_qwen3_8b",
}
LOCAL_OUTPUT_ROOTS = {
    arm: Path("/content/value-misalignment-runs") / OUTPUT_SLUGS[arm]
    for arm in EVALUATION_ARMS
}
DRIVE_OUTPUT_ROOTS = {
    arm: Path("/content/drive/MyDrive/value-misalignment") / OUTPUT_SLUGS[arm]
    for arm in EVALUATION_ARMS
}
FORCE_RETRAIN = False
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"

def config_for_arm(arm):
    return DilemmaSFTConfig(
        output_root=LOCAL_OUTPUT_ROOTS[arm],
        training_arm=arm,
        dataset_path=DEFAULT_DATASET_PATHS[arm],
        base_model="Qwen/Qwen3-8B",
        model_revision="b968826d9c46dd6066d109eabc6255188de91218",
        max_length=1024,
        num_train_epochs=3,
        learning_rate=1e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        lora_rank=16,
        lora_alpha=32,
        lora_dropout=0.05,
        eval_batch_size=4,
        seed=42,
        cost_counts=(0, 1, 10, 100, 1_000, 10_000, 100_000, 1_000_000),
    )

CONFIGS = {arm: config_for_arm(arm) for arm in EVALUATION_ARMS}
CONFIG = CONFIGS[TRAINING_ARM]

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add a Colab secret named GITHUB_TOKEN and grant this notebook access before training."
        ) from exc
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN is empty; publication is enabled and required by this workflow."
        )
print("Arm selected only for optional forced retraining:", CONFIG.training_arm)
print("Evaluation arms:", EVALUATION_ARMS)
print("N values:", CONFIG.cost_counts)
CONFIGS

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
from scripts.ecological_prompt_sft import load_training_examples

dataset_rows = []
for arm, arm_config in CONFIGS.items():
    examples, manifest = load_training_examples(
        arm_config.dataset_path,
        training_arm=arm,
    )
    assert len(examples) == 98
    assert manifest["training_arm"] == arm
    assert manifest["contains_normative_labels"] == (arm != "prompt_only")
    assert manifest["contains_assistant_responses"] == (arm != "prompt_only")
    dataset_rows.append({
        "arm": arm,
        "examples": len(examples),
        "records_sha256": manifest["records_sha256"],
        "assistant_target": manifest.get("assistant_target_field"),
    })
    example = examples[0]
    preview = f"### `{arm}` — `{example['id']}`\n\n**User**\n\n{example['dilemma']}"
    if "assistant_answer" in example:
        preview += f"\n\n**Assistant**\n\n{example['assistant_answer']}"
    display(Markdown(preview))
display(pd.DataFrame(dataset_rows))

In [ ]:
from scripts.ecological_prompt_sft import (
    find_complete_runs_for_arms,
    persist_run_to_colab_drive,
    run_dilemma_sft,
)

if FORCE_RETRAIN:
    print(f"Starting {CONFIG.training_arm} Qwen fine-tuning on all 98 dilemmas.")
    local_artifacts = run_dilemma_sft(CONFIG)
    retrained = persist_run_to_colab_drive(
        local_artifacts,
        DRIVE_OUTPUT_ROOTS[TRAINING_ARM],
    )
    print("Completed and freshly verified Drive run:", retrained.run_dir)

print("Discovering all three compatible checkpoints on Drive...")
artifacts_by_arm = find_complete_runs_for_arms(DRIVE_OUTPUT_ROOTS, CONFIGS)
artifacts = artifacts_by_arm[TRAINING_ARM]
for arm, arm_artifacts in artifacts_by_arm.items():
    print(f"VERIFIED {arm}: {arm_artifacts.run_dir}")

In [ ]:
import json
import pandas as pd

verified_run_rows = []
for arm, arm_artifacts in artifacts_by_arm.items():
    complete = json.loads(arm_artifacts.complete_marker_path.read_text())
    train_metrics = json.loads(arm_artifacts.train_metrics_path.read_text())
    dataset_manifest = json.loads(arm_artifacts.dataset_manifest_path.read_text())
    run_metadata = json.loads(arm_artifacts.metadata_path.read_text())
    assert complete["status"] == "complete"
    assert dataset_manifest["example_count"] == 98
    assert dataset_manifest.get("training_arm", "prompt_only") == arm
    assert train_metrics.get("training_arm", "prompt_only") == arm
    verified_run_rows.append({
        "arm": arm,
        "training_objective": run_metadata["training_objective"],
        "run": arm_artifacts.run_dir.name,
        "adapter_revision": run_metadata["resolved_revisions"]["final_adapter_sha256"],
        "supervised_token_mean": dataset_manifest["tokenization"]["supervised_tokens"]["mean"],
    })
display(pd.DataFrame(verified_run_rows))

In [ ]:
from scripts.ecological_prompt_sft import (
    EXTREME_V2_TEMPLATES,
    build_extreme_v2_cases,
)

primary_preview_cases = build_extreme_v2_cases(CONFIG.cost_counts)
assert len(primary_preview_cases) == len(EXTREME_V2_TEMPLATES) * len(CONFIG.cost_counts)
print(f"Reviewing all {len(primary_preview_cases)} primary cases before inference.")
for case in primary_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
# Legacy six-control preview, retained but demoted from the default workflow.
# from scripts.ecological_prompt_sft import (
#     EXTREME_V2_CONTROL_TEMPLATES,
#     build_extreme_v2_control_cases,
# )
# control_preview_cases = build_extreme_v2_control_cases(CONFIG.cost_counts)
# assert len(control_preview_cases) == 4 * len(CONFIG.cost_counts) + 2
# for case in control_preview_cases:
#     display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
from scripts.ecological_prompt_sft import run_extreme_v2_workflow

primary_workflow = run_extreme_v2_workflow(
    artifacts,
    cost_counts=CONFIG.cost_counts,
    batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
if primary_workflow.evaluation_reused:
    print("PRIMARY EVALUATION SKIPPED — reusing verified Drive results.")
else:
    print("PRIMARY EVALUATION COMPLETE — verified after a fresh Drive remount.")
primary_workflow.validation

In [ ]:
from IPython.display import Image

primary_eval = primary_workflow.evaluation_artifacts
primary_scores = pd.read_csv(primary_eval.raw_scores_path)
assert len(primary_scores) == 2 * len(primary_preview_cases)
assert set(primary_scores["model_role"]) == {"base", "aligned"}
display(primary_scores.pivot(
    index=["template", "cost_count"],
    columns="model_role",
    values=["p_implement", "semantic_logit_implement"],
).sort_index())
display(pd.read_csv(primary_eval.thresholds_path))
display(Image(filename=str(primary_eval.plot_path)))
print("Verified primary Drive bundle:", primary_eval.output_dir)

In [ ]:
# Legacy six-control evaluation, retained but not run by default.
# from scripts.ecological_prompt_sft import run_extreme_v2_control_workflow
# control_workflow = run_extreme_v2_control_workflow(
#     artifacts,
#     cost_counts=CONFIG.cost_counts,
#     batch_size=CONFIG.eval_batch_size,
#     force_evaluation=FORCE_EVALUATION,
# )
# control_workflow.validation

In [ ]:
# Legacy six-control display, retained but not run by default.
# control_eval = control_workflow.evaluation_artifacts
# control_scores = pd.read_csv(control_eval.raw_scores_path)
# display(control_scores)
# display(pd.read_csv(control_eval.thresholds_path))
# display(Image(filename=str(control_eval.plot_path)))
# print("Verified control Drive bundle:", control_eval.output_dir)

## Supervision-matched readout battery

Every readout retains the original eight `extreme_v2` scenario bodies and all eight cost values. Reversed `Yes`/`No` asks about the human-protective action, so `No` is mapped back to the ecological option. Counterbalanced `A`/`B` presents both option orders. Complete option-text scoring also presents both orders and scores each entire option; its primary margin uses mean log-probability per candidate token, while the raw summed sequence log-probabilities and token counts remain in the CSV. In every case, positive `semantic_logit_implement` and larger `p_implement` mean greater support for the ecological option.

In [ ]:
from scripts.ecological_prompt_sft import build_supervision_matched_readout_cases

readout_preview_cases = build_supervision_matched_readout_cases(CONFIG.cost_counts)
assert len(readout_preview_cases) == 320
assert len({case['case_id'] for case in readout_preview_cases}) == 320
display(pd.DataFrame(readout_preview_cases).groupby(
    ['readout_type', 'readout_variant']
).size().rename('case_count').reset_index())
print("Reviewing one cost-1 prompt for each of the 40 readout templates.")
for case in readout_preview_cases:
    if case['cost_count'] == 1:
        display(Markdown(
            f"### `{case['case_id']}`\n\n"
            f"Ecological candidate: `{case['candidate_implement']}`  \n"
            f"Human candidate: `{case['candidate_reject']}`  \n\n{case['prompt']}"
        ))

In [ ]:
from scripts.ecological_prompt_sft import run_supervision_matched_readout_workflow

readout_workflows = {}
for arm in EVALUATION_ARMS:
    print(f"\n=== Evaluating {arm} ===")
    workflow = run_supervision_matched_readout_workflow(
        artifacts_by_arm[arm],
        cost_counts=CONFIGS[arm].cost_counts,
        batch_size=CONFIGS[arm].eval_batch_size,
        force_evaluation=FORCE_EVALUATION,
    )
    readout_workflows[arm] = workflow
    status = "REUSED" if workflow.evaluation_reused else "COMPLETED"
    print(f"{arm}: {status} and hash-verified on Drive")
    print(workflow.validation)

In [ ]:
readout_evals = {}
for arm, workflow in readout_workflows.items():
    evaluation = workflow.evaluation_artifacts
    readout_evals[arm] = evaluation
    scores = pd.read_csv(evaluation.raw_scores_path)
    assert len(scores) == 640
    assert set(scores['model_role']) == {'base', 'aligned'}
    paired = scores.pivot(
        index=['case_id', 'readout_type', 'readout_variant'],
        columns='model_role',
        values='semantic_logit_implement',
    ).reset_index()
    paired['aligned_minus_base'] = paired['aligned'] - paired['base']
    summary = paired.groupby(
        ['readout_type', 'readout_variant']
    )[['base', 'aligned', 'aligned_minus_base']].mean().reset_index()
    display(Markdown(f"## {arm}"))
    display(summary)
    display(pd.read_csv(evaluation.thresholds_path))
    display(Image(filename=str(evaluation.plot_path)))
    print("Verified Drive bundle:", evaluation.output_dir)

In [ ]:
from scripts.ecological_prompt_sft import publish_results_to_github

if PUBLISH_TO_GITHUB:
    readout_publications = {}
    for arm, evaluation in readout_evals.items():
        publication = publish_results_to_github(
            evaluation,
            source_run_name=artifacts_by_arm[arm].run_dir.name,
            github_repository=GITHUB_REPOSITORY,
            branch=GITHUB_BRANCH,
            github_token=GITHUB_TOKEN,
            repo_root=REPO_DIR,
        )
        readout_publications[arm] = publication
        print(f"{arm} GitHub publication verified: {publication.html_url}")
else:
    print("GitHub publication disabled; all verified Drive bundles remain intact.")